In [ ]:
library(pacman)
p_load(plyr,tidyverse,skimr,recipes,rsample,caret,doParallel)

In [ ]:
list.files('../input')

In [ ]:
tr = read_csv('../input/cs-training.csv') %>% rename(Id = X1)
te = read_csv('../input/cs-test.csv') %>% rename(Id = X1)

In [ ]:
skim_to_list(tr)

#### simple model

In [ ]:
data_prep = function(x) x %>% select_all(tolower) %>% 
rename(target=seriousdlqin2yrs) %>% 
mutate(target = as.character(target),
       target = recode(target,'0'='good','1'='bad')) %>% 
mutate(target = as.factor(target))

In [ ]:
set.seed(7)
tr = data_prep(tr) %>% sample_n(30000)
te = data_prep(te)

In [ ]:
names(tr)

In [ ]:
rec<-recipe(~.,data=tr) %>% 
add_role(id,new_role='id variable') %>% 
add_role(target,new_role = 'outcome') %>% 
add_role(-id,-target,new_role='predictor') %>% 
step_meanimpute(all_predictors()) %>% 
step_center(all_predictors()) %>% 
step_scale(all_predictors()) %>% 
step_zv(all_predictors()) %>% 
prep(retain=T)

In [ ]:
tr1 = juice(rec)
te1 = bake(rec,te)

In [ ]:
set.seed(3)
kfold = createFolds(tr1$target,k=5)

ranger::ranger에서 
1. target은 숫자형이면 에러남  즉, 0/1 은 'good'/'bad'로 바꿔줘야 함  
2. predict에서 type='prob' 쓰려면 trainControl에서 classProbs=TRUE 지정해줘야 함  
3. variable importance 쓰려면 importance='impurity' 등으로 지정해 줘야 함 

In [ ]:
ctrl = trainControl(index=kfold,classProbs=TRUE)

In [ ]:
tr2<-rename(tr1,num3059 = `numberoftime30-59dayspastduenotworse`,
           num6089 = `numberoftime60-89dayspastduenotworse`)

te2<-rename(te1,num3059 = `numberoftime30-59dayspastduenotworse`,
           num6089 = `numberoftime60-89dayspastduenotworse`)

In [ ]:
count(tr2,target)

registerDoParallel(4)

fit.rf = train(target~.,data=select(tr1,-id),
           method='ranger',
           num.trees=300,
           trControl=ctrl,
           tuneLength=3,
           importance='impurity',
           metric='Kappa')
print('rf done')
fit.gbm = train(target~.,data=select(tr1,-id),
               method='gbm',
               trControl=ctrl,
               tuneLength=3,
               verbose=FALSE,
               metric='Kappa')
print('gbm done')

stopImplicitCluster()

In [ ]:
cl <- makePSOCKcluster(5)
registerDoParallel(cl)

fit.gbm = train(target~.,data=select(tr1,-id),
               method='gbm',
               trControl=ctrl,
               tuneLength=3,
               verbose=FALSE,
               metric='Kappa')

stopCluster(cl)

rsmp = resamples(list(rf=fit.rf,gbm=fit.gbm))

summary(rsmp)

In [ ]:
pred_prob = predict(fit.gbm,te1,type='prob')

In [ ]:
gmc_sumit<-data_frame(Id=te1$id,Probability=pred_prob[,2]) %>% 
mutate(Probability=as.double(Probability),
      Id = as.integer(Id))

In [ ]:
write_csv(gmc_sumit,'gbm_summit.csv')